# LAB 08 — Tokenization and Attention Mask

Notebook này minh họa pipeline từ raw text đến `input_ids`, `attention_mask` và biểu diễn ngữ cảnh của token `[CLS]`. Không huấn luyện model trong notebook này.

## 1. Khởi tạo tokenizer

Tokenizer phải tương ứng với pretrained Transformer được dùng ở các thí nghiệm sau.

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Special tokens:", tokenizer.special_tokens_map)

## 2. Tokenize một câu

Theo yêu cầu đề bài, kiểm tra các khóa, token sequence, token IDs và attention mask.

In [ ]:
text = "The cats are playing."

tokens = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt",
)

print("tokens.keys():", list(tokens.keys()))
print("input_ids:", tokens["input_ids"])
print("attention_mask:", tokens["attention_mask"])

token_ids = tokens["input_ids"][0].tolist()
token_sequence = tokenizer.convert_ids_to_tokens(token_ids)
print("token sequence:", token_sequence)

In [ ]:
print("Special token IDs:")
for name, token in tokenizer.special_tokens_map.items():
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"{name}: {token!r} -> {token_id}")

print("\nToken and mask:")
for token, token_id, mask_value in zip(
    token_sequence,
    tokens["input_ids"][0].tolist(),
    tokens["attention_mask"][0].tolist(),
):
    print(f"{token:>12} | id={token_id:>6} | attention={mask_value}")

Trong attention mask, `1` là vị trí token hợp lệ; `0` là vị trí padding. Các token đặc biệt thường gặp là `[CLS]` (đại diện sequence), `[SEP]` (phân tách sequence), `[PAD]` (padding), `[MASK]` (MLM) và `[UNK]` (token không nhận diện được, tùy tokenizer).

In [ ]:
words = text.rstrip(".").split() + ["."]
active_tokens = [
    token
    for token, mask_value in zip(
        token_sequence,
        tokens["attention_mask"][0].tolist(),
    )
    if mask_value == 1
]

print("Words:", words)
print("Số words:", len(words))
print("Số tokens hợp lệ:", len(active_tokens))
print("Số subword tokens trong nội dung:", sum(
    1 for token in active_tokens if not token.startswith("[")
))
print("Lưu ý: một word có thể tách thành nhiều subword, ví dụ các token có tiền tố ##.")

## 3. Biểu diễn `[CLS]` trong Transformer

Classification head dùng vector đầu tiên của `last_hidden_state`, tức `last_hidden_state[:, 0, :]`, làm biểu diễn cho toàn bộ sequence.

In [ ]:
import torch

model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

with torch.no_grad():
    outputs = model(**tokens)

last_hidden_state = outputs.last_hidden_state
cls_representation = last_hidden_state[:, 0, :]

print("last_hidden_state shape:", tuple(last_hidden_state.shape))
print("[CLS] representation shape:", tuple(cls_representation.shape))